在深度学习中，优化器的核心任务是通过计算损失函数的梯度来更新模型参数，从而使损失最小化。经过多年的发展，优化器演化出了多个家族，从最基础的梯度下降逐步发展到自适应学习率和二阶优化器。

以下是深度学习中主流和经典优化器的完整分类盘点：

---

## 1. 经典梯度下降家族 (Gradient Descent)

这类优化器使用固定的或全局统一的学习率，完全依赖当前或历史梯度的方向进行更新。

* **BGD (Batch Gradient Descent / 批量梯度下降)**：每次更新使用整个训练集计算梯度。极其准确但在大数据量下内存崩溃、计算极慢。
* **SGD (Stochastic Gradient Descent / 随机梯度下降)**：每次更新仅随机选择一个样本计算梯度。速度快，但由于引入了噪声，更新轨迹呈锯齿状，极易陷入局部最优或鞍点。
* **MBGD (Mini-batch Gradient Descent / 小批量梯度下降)**：结合前两者，每次使用一小批（Batch Size）样本。**这就是我们今天常说的标准 "SGD"**。
* **SGD with Momentum (带动量的SGD)**：引入了物理学中的“惯性”概念（一阶动量）。它累加了历史梯度，在梯度方向一致的地方加速，在方向震荡的地方减速，能有效摆脱局部最优并加速收敛。
* **NAG (Nesterov Accelerated Gradient / 牛顿加速梯度)**：动量 SGD 的升级版。它在计算当前梯度时，先顺着动量方向“往前看一步”，在预估的未来位置计算梯度。这种“前瞻性”使得更新更加敏锐，防止冲过头。

---

## 2. 自适应学习率家族 (Adaptive Learning Rate)

这类优化器的核心思想是**为每个参数定制独立且动态调整的学习率**。通常频繁更新的参数学习率变小，稀疏更新的参数学习率保持较大。

* **AdaGrad (Adaptive Gradient)**：自适应学习率的开山之作。它通过累加历史梯度的平方来缩放当前学习率。缺点是后期分母的分母会无限变大，导致学习率过早“枯竭”，模型停止学习。
* **RMSprop (Root Mean Square Propagation)**：为了解决 AdaGrad 的枯竭问题，Hinton 提出了 RMSprop。它不再无限累加历史梯度，而是采用**指数移动平均（EMA）**，只关注最近一段时间的梯度，成功让学习率保持活力。
* **AdaDelta**：与 RMSprop 几乎同时提出，同样使用 EMA 解决 AdaGrad 的问题，但它更进一步，去除了超参数学习率 $\alpha$，完全通过参数自身单位的性质来计算更新步长。
* **Adam (Adaptive Moment Estimation)**：深度学习中最著名的“万金油”优化器。它**集大成于一身**：结合了 **Momentum（一阶动量，控制方向）** 和 **RMSprop（二阶动量，控制步长）**，并对初始化偏差进行了修正。通常是大多数模型的默认首选。

---

## 3. 现代与变体家族 (Modern & Variant Optimizers)

随着大模型（LLM、Vision Transformer）的兴起，经典的 Adam暴露出一些短板（如泛化能力稍逊于 SGD、显存占用高、在特定结构上不收敛等），催生了一批现代优化器。

* **AdamW (Adam with Decoupled Weight Decay)**：**目前大模型（如 Transformer/GPT）训练的绝对主力**。经典的 Adam 在处理 $L_2$ 正则化时，会将权重衰减（Weight Decay）直接混入梯度中，导致自适应步长计算失真。AdamW 将权重衰减与梯度更新**完全解耦**，大幅提升了模型的泛化能力。
* **AMSGrad**：修正了 Adam 的一个理论漏洞。在某些情况下，Adam 的二阶动量可能导致学习率单调递增从而引发不收敛，AMSGrad 通过强制保留历史最大的二阶动量来确保学习率单调递减。
* **AdaBound**：旨在结合 SGD 的高泛化能力和 Adam 的快速收敛。它在训练初期像 Adam 一样快，随着训练进行，动态将学习率限制在一定区间内，后期逐渐平滑演变为 SGD。
* **Nadam / Radam (Rectified Adam)**：
* **Nadam**：将 Nesterov 动量融入 Adam。
* **RAdam**：引入了解调机制，自动根据方差动态调整学习率，使得模型在训练初期不需要极其小心的 Learning Rate Warm-up（预热）。


* **Lion (Evolved Sign Momentum)**：谷歌通过进化算法自动搜索出来的优化器。它**只保存梯度的符号（Sign，即 $+1$ 或 $-1$）**，不仅更新速度快，而且因为不需要存储高精度的二阶动量状态，能大幅**节省显存**，在大模型训练中非常受欢迎。

---

## 4. 二阶优化器家族 (Second-Order Optimizers)

前述优化器都属于“一阶优化”，即只考虑梯度（一阶导数）。二阶优化器同时考虑海森矩阵（Hessian Matrix，二阶导数），即考虑梯度的变化率（曲率），因此能做到“一步到位”，理论上收敛步数极少。

* **Newton's Method (牛顿法)**：直接计算 Hessian 矩阵的逆。计算复杂度为 $O(N^3)$（$N$ 为参数量），在数亿参数的深度学习中完全不可行。
* **BFGS / L-BFGS**：拟牛顿法。L-BFGS 限制了内存占用，不需要显式存储巨大的 Hessian 矩阵，在某些小体量、高精度的科学计算（如 PINN 物理神经网络）或传统机器学习（如逻辑回归）中很常用。
* **K-FAC (Kronecker-factored Approximate Curvature)**：对 Hessian 矩阵进行克罗内克积近似分解，大幅降低了计算成本，是目前少数能在中等规模深度学习网络中跑通的二阶优化器。

---

## 5. 分布式与超大模型专属家族

针对千亿参数规模、数百台机器分布式训练衍生出的特殊优化器。

* **LAMB (Layer-wise Adaptive Moments optimizer for Batch training)**：专为超大 Batch Size 训练设计的优化器（如用成千上万张 GPU 同时训练 BERT）。它在 Layer 级别做自适应归一化，能保证在大 Batch 训练时梯度不崩塌。
* **LARS (Layer-wise Adaptive Rate Scaling)**：与 LAMB 类似，也是 Layer 级别的自适应，但基于 SGD 架构，主要用于大规模自监督视觉模型（如 SimCLR）的巨量 Batch 训练。
* **FusedOptimizers (如 DeepSpeed/Apex 中的 FusedAdam)**：将优化器的多步算子合并为一个 GPU Kernel 执行，不改变算法逻辑，但大幅提升硬件吞吐、降低显存抖动。

---

## 💡 总结与选型指南

| 场景需求 | 推荐优化器 | 原因 |
| --- | --- | --- |
| **通用/无脑首选** | `AdamW` | 结合了自适应步长与正确的权重衰减，目前工业界综合表现最稳。 |
| **极致泛化/精调** | `SGD with Momentum` | 很多 CV 经典网络（如 ResNet）到最后冲刺高精度时，SGD 的泛化表现往往依然强于 Adam 家族。 |
| **大语言模型/LLM** | `AdamW` / `Lion` | AdamW 是绝对标配；Lion 常用于追求显存优化和训练加速的场景。 |
| **物理/科学计算神经网络** | `L-BFGS` | 面对需要高精度二阶连续性的复杂动力学方程，二阶优化器更具优势。 |